# Modelltraining und Evaluierung der Busauslastungsprognose

Dieses Notebook beschreibt den vollständigen Workflow zur Vorhersage der Busauslastung mithilfe von Ensemble-Lernverfahren unter Verwendung von Spark, XGBoost und Ridge Regression.

## 1. Installations- und Umgebungsanforderungen


In [ ]:
# Installieren benötigter Pakete
# !pip install tqdm

## 2. Imports und Bibliotheken

In [ ]:
# Import wesentlicher Bibliotheken für Datenmanipulation, Modellierung und Evaluation
import time
import numpy as np
import pandas as pd
import ast
from itertools import product
from concurrent.futures import ThreadPoolExecutor, as_completed
import shap
import matplotlib.pyplot as plt
import fsspec
import io
import joblib

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import io, joblib, fsspec
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from tqdm import tqdm

In [ ]:
# Definition des Szenarios (Modus der Modellpipeline)
SZENARIO = "3"

In [ ]:
# Erstellen bzw. Abrufen einer Spark-Session für Datenzugriff und -verarbeitung
spark = SparkSession.builder.getOrCreate()

## 5. Pfadkonstanten

In [ ]:
PARQUET_PATH = "data/enriched_data"
BASE_TUNING_PATH  = "machine_learning/hyperparameter_tuning/optimal_parameters"


## 6. Definition der Feature-Listen

In [ ]:
# JSON-Parameter zur dynamischen Feature-Auswahl
feature_list_json = [
    "weekday","month","is_day","direction","stop_name","lag_stop_name","lead_stop_name",
    "temperature_2m","rain","weighted_avg_rain","rain_bins","snowfall","wind_speed_10m",
    "is_public_holiday","is_school_holiday","event_type"
]
# Kategorische Merkmale für One-Hot-Encoding
cat_cols = [
    "stop_name","direction","lag_stop_name","lead_stop_name",
    "rain_bins","event_type","is_public_holiday","is_school_holiday"
]

## 7. Erzeugung des Parameter-Rasters für Teilmodelle

In [ ]:
# Linien, gemeinsame Parameter und Szenario-spezifische Targets
lines = ["5", "6", "11"]
common = {
    "COVID_FILTER": "Post-COVID",
    "ALGORITHM":    "GradientBoosting",
    "ITERATION":    "2"
}
if SZENARIO == "3":
    targets = ["boarders", "alighters"]
else:
    targets = ["boarders", "alighters", "occupancy_s-1"]

time_aggs = ["5", "15", "30", "60"]
# Kombinatorische Erzeugung aller Parameter-Konstellationen
param_iterations = [
    {"BUS_LINE": bl, "TARGET": tgt, "TIME_AGGREGATION": ta, **common}
    for bl, tgt, ta in product(lines, targets, time_aggs)
]

## 8. Laden und Vorverarbeiten der Daten

In [ ]:
# Speicherung der geladenen DataFrames pro Parameter-Kombination
df_store = {}
for params in param_iterations:
    bl, tgt, ta = params["BUS_LINE"], params["TARGET"], params["TIME_AGGREGATION"]
    # Bestimmung dynamischer Features je nach Zielgröße
    if tgt == "occupancy_s-1":
        dynamic = [
            f"interval_{ta}min", f"bus_count_interval_{ta}min",
            f"avg_occupancy_s-1_interval_{ta}min", f"lag_occupancy_s-1_interval_{ta}min"
        ]
        if SZENARIO == "2":
            dynamic += ["occupancy_s-2", "delay_s-2"]
    else:
        dynamic = [
            f"interval_{ta}min", f"bus_count_interval_{ta}min",
            f"avg_{tgt}_interval_{ta}min", f"lag_{tgt}_interval_{ta}min",
            "deviation", "stop_skipped"
        ]
    FEATURE_SET = [f for f in feature_list_json if f not in dynamic] + dynamic + [tgt]
    select_cols = FEATURE_SET + ["capacity", "occupancy"]
    if SZENARIO == "3":
        select_cols.append("occupancy_s-1")
    # Laden, Filtern, Ordern und Konvertieren in Pandas
    pdf = (
        spark.read.parquet(PARQUET_PATH)
             .filter(col("covid_period") == params["COVID_FILTER"])
             .filter(col("line") == bl)
             .orderBy("actual_arrival")
             .select(*select_cols)
             .limit(100)
             .toPandas()
    )
    df_store[(bl, tgt, ta)] = pdf

## 9. Training der Teilmodelle (XGBoost)

In [ ]:
cache_store = {}
sub_records = []

def train_and_cache(params):
    bl, tgt, ta = params["BUS_LINE"], params["TARGET"], params["TIME_AGGREGATION"]
    pdf = df_store[(bl, tgt, ta)]
    # Kapazität und tatsächliche Auslastung
    cap, occ = pdf["capacity"].values, pdf["occupancy"].values
    # One-Hot-Encoding und Feature-Matrix
    df_ohe = pd.get_dummies(pdf.drop(columns=["capacity", "occupancy"]), columns=cat_cols)
    X = df_ohe.drop(columns=[tgt]); y = df_ohe[tgt]
    # Deterministischer 80/20-Split
    cut = int(len(X) * 0.8)
    X_tr, X_te = X.iloc[:cut], X.iloc[cut:]
    y_tr, y_te = y.iloc[:cut], y.iloc[cut:]
    true_util_te = np.where(cap[cut:]==0, 0.0, occ[cut:]/cap[cut:])
    # Auswahl des Tuning-Ordners
    if SZENARIO == "3" or (tgt == "occupancy_s-1" and SZENARIO == "2"):
        tuning_folder = "hyperparameter_tuning_szenario2/"
    else:
        tuning_folder = "hyperparameter_tuning_szenario1/"
    file_path = BASE_TUNING_PATH + tuning_folder + f"XGB_RS_{bl}_{tgt}_{ta}_szenario{SZENARIO}.csv"
    df_hp = pd.read_csv(file_path, dtype=str)
    best_params = ast.literal_eval(df_hp.loc[0,'best_params']); best_params.setdefault("random_state",42)
    # Modellinitialisierung, Training und Inferenz
    model = XGBRegressor(**best_params, objective="reg:squarederror", n_jobs=-1)
    t0 = time.time(); model.fit(X_tr,y_tr); t_tr = time.time()-t0
    t1 = time.time(); pred_cnt = model.predict(X_te); t_inf = time.time()-t1
    pred_util_te = np.clip(pred_cnt/cap[cut:],0,1)
    # Metrikenberechnung
    mae = mean_absolute_error(true_util_te,pred_util_te)
    rmse = mean_squared_error(true_util_te,pred_util_te,squared=False)
    r2 = r2_score(true_util_te,pred_util_te)
    mape = np.mean(np.abs((true_util_te-pred_util_te)/np.where(true_util_te==0,1e-8,true_util_te)))*100
    smape = np.mean(np.abs(pred_util_te-true_util_te)/(np.maximum(np.abs(true_util_te)+np.abs(pred_util_te),1e-8)/2))*100
    # Caching für Meta-Schritt
    cache_store[(bl,tgt,ta,params["ALGORITHM"],params["ITERATION"])] = {
        "model":model,"X_test":X_te,"true_util":true_util_te,
        "pred_util":pred_util_te,"feature_names":X.columns.tolist()
    }
    return {"BUS_LINE":bl,"TARGET":tgt,"TIME_AGGREGATION":ta,
            "train_time_s":t_tr,"infer_time_s":t_inf,
            "MAE_util":mae,"RMSE_util":rmse,"R2_util":r2,
            "MAPE_util":mape,"sMAPE_util":smape,
            "predictions":pred_util_te.tolist(),"used_params":best_params}

# Paralleles Training aller Submodelle
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(train_and_cache,p) for p in param_iterations]
    for rec in tqdm(as_completed(futures),total=len(futures),desc="Submodelle trainieren"):
        sub_records.append(rec.result())
results_df = pd.DataFrame(sub_records)


## 10. Meta-Holdout Evaluation (Ridge Regression)

In [ ]:
def meta_holdout_eval(bus_line,time_agg,n_splits=5):
    '''
    Meta-Evaluierung mit Ridge Regression unter Verwendung der Teilmodell-Vorhersagen.
    '''
    grp = results_df[(results_df['BUS_LINE']==bus_line)&(results_df['TIME_AGGREGATION']==time_agg)]
    sub_train_time = grp['train_time_s'].sum()
    preds = {row['TARGET']:np.array(row['predictions']) for _,row in grp.iterrows()}
    pdf_occ = df_store[(bus_line,'occupancy_s-1',time_agg)]
    occ,cap = pdf_occ['occupancy'].values,pdf_occ['capacity'].values
    cutoff=int(len(occ)*0.8)
    y_meta=np.where(cap[cutoff:]==0,0.0,occ[cutoff:]/cap[cutoff:])
    prev_util=np.where(cap==0,0.0,pdf_occ['occupancy_s-1'].values/cap)
    if SZENARIO=="3":
        X_meta=np.vstack([preds['boarders'],-preds['alighters'],prev_util[cutoff:]]).T
        sub_keys=['boarders','alighters']
    else:
        X_meta=np.vstack([preds['occupancy_s-1'],preds['boarders'],-preds['alighters']]).T
        sub_keys=['occupancy_s-1','boarders','alighters']
    tscv=TimeSeriesSplit(n_splits=n_splits)
    maes=[]; rmses=[]; r2s=[]; mapes=[]; smapes=[]
    meta_train_time=meta_infer_time=0.0
    for train_idx,test_idx in tscv.split(X_meta):
        X_tr,X_te=X_meta[train_idx],X_meta[test_idx]
        y_tr,y_te=y_meta[train_idx],y_meta[test_idx]
        model=Ridge()
        t0=time.time(); model.fit(X_tr,y_tr); meta_train_time+=time.time()-t0
        t1=time.time(); y_pred=np.clip(model.predict(X_te),0,1); meta_infer_time+=time.time()-t1
        maes.append(mean_absolute_error(y_te,y_pred))
        rmses.append(mean_squared_error(y_te,y_pred,squared=False))
        r2s.append(r2_score(y_te,y_pred))
        mapes.append(np.mean(np.abs((y_te-y_pred)/np.where(y_te==0,1e-8,y_te)))*100)
        smapes.append(np.mean(np.abs(y_pred-y_te)/(np.maximum(np.abs(y_te)+np.abs(y_pred),1e-8)/2))*100)
    sub_models={t:cache_store[(bus_line,t,time_agg,common['ALGORITHM'],common['ITERATION'])]['model'] for t in sub_keys}
    return {'BUS_LINE':bus_line,'TIME_AGGREGATION':time_agg,
            'sub_train_time_s':sub_train_time,'meta_train_time_s':meta_train_time,
            'meta_infer_time_s':meta_infer_time,
            'total_train_time_s':sub_train_time+meta_train_time,
            'MAE_meta':np.mean(maes),'RMSE_meta':np.mean(rmses),
            'R2_meta':np.mean(r2s),'MAPE_meta':np.mean(mapes),
            'sMAPE_meta':np.mean(smapes),'meta_model':model,
            'sub_models':sub_models}

## 11. Ausführen der Meta-Holdout-Evaluierung und Persistierung der Modelle

In [ ]:
# 1) Kombinationen der Linien und Aggregationszeiträume bestimmen
combos = results_df[['BUS_LINE', 'TIME_AGGREGATION']] \
             .drop_duplicates() \
             .to_records(index=False)

# 2) Meta-Holdout-Evaluation parallel ausführen
meta_results = []
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {
        executor.submit(meta_holdout_eval, bl, ta): (bl, ta)
        for bl, ta in combos
    }
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Meta-Modelle"):
        res = fut.result()
        if res is not None:
            meta_results.append(res)

# 3) Ergebnisse in DataFrame packen
meta_results_df = pd.DataFrame(meta_results)

# 4) Modell-Bundles für die besten Kombinationen erstellen und speichern
MODEL_DIR = "machine_learning/models/"
best_rows = meta_results_df.loc[
    meta_results_df.groupby('BUS_LINE')['RMSE_meta'].idxmin()
].reset_index(drop=True)

for _, row in best_rows.iterrows():
    bl = row['BUS_LINE']
    ta = row['TIME_AGGREGATION']
    bundle = {'meta_model': row['meta_model']}
    # Untermodule automatisch hinzufügen
    for tgt, model in row['sub_models'].items():
        bundle[f"{tgt}_model"] = model

    filename = f"bundle_ridge_{bl}_agg{ta}_{SZENARIO}.joblib"
    remote_path = MODEL_DIR + filename

    buf = io.BytesIO()
    joblib.dump(bundle, buf)
    buf.seek(0)
    with fsspec.open(remote_path, 'wb') as f_out:
        f_out.write(buf.read())

    print(f"✅ Bundle gespeichert: {remote_path}")

## 12. Best-Per-Line-Auswahl und Speicherung

In [ ]:
# Auswahl der besten Kombination je Bus-Linie basierend auf RMSE_meta
best_per_line = (
    meta_results_df.loc[
        meta_results_df.groupby("BUS_LINE")["RMSE_meta"].idxmin()
    ]
    .reset_index(drop=True)
)

# Speichern der Metriken als CSV (unverändert, auskommentiert)
# BEST_PATH = BASE_PATH + f"ergebnisse/metriken/best_per_line_GB_{SZENARIO}.csv"
# best_per_line.to_csv(BEST_PATH, index=False)

## 13. SHAP-Werte für Teilmodelle berechnen und speichern

In [ ]:
# Erzeugen und Speichern der SHAP-Werte für alle Submodelle im Szenario 1
if SZENARIO == "1":
    ALG = "GradientBoosting"; IT = "2"; MAX_SAMPLES = 1000
    shap_dfs = []

    # Für jede Bus-Linie und Aggregation die drei Submodelle
    for _, row in best_per_line.iterrows():
        bl, ta = row["BUS_LINE"], row["TIME_AGGREGATION"]
        for tgt in ("occupancy_s-1", "boarders", "alighters"):  
            key = (bl, tgt, ta, ALG, IT)
            entry = cache_store[key]
            model = entry["model"]
            Xc_full = entry["X_test"]
            feats = entry["feature_names"]

            # Zufälliges Sampling der Testdaten für SHAP-Analyse
            Xc = Xc_full.sample(n=min(MAX_SAMPLES, Xc_full.shape[0]), random_state=42)

            # Berechnung der SHAP-Werte
            explainer = shap.TreeExplainer(model)
            sv = explainer.shap_values(Xc)

            # Mittlere absolute SHAP-Werte und Prozentanteil
            mean_abs = np.abs(sv).mean(axis=0)
            pct = mean_abs / mean_abs.sum() * 100

            shap_dfs.append(pd.DataFrame({
                "BUS_LINE":         bl,
                "TIME_AGGREGATION": ta,
                "TARGET":           tgt,
                "feature":          feats,
                "mean_abs_shap":    mean_abs,
                "percent_shap":     pct
            }))

    shap_results_df = pd.concat(shap_dfs, ignore_index=True)
    # Optional: Speichern der SHAP-Ergebnisse als CSV im Data Lake
    SHAP_CSV_PATH = BASE_PATH + "ergebnisse/shap/shap_results_submodels_{Szenario}.csv"
    with fsspec.open(SHAP_CSV_PATH, "w") as f:
         shap_results_df.to_csv(f, index=False)
    print(f"✔ SHAP-Werte für alle Submodelle wurden erzeugt und in shap_results_df gespeichert.")